# 校园共享空间姿态、个体预测与人车图

新建私有 Kaggle Notebook 后导入本文件，选择 GPU T4，打开 Internet，再从上至下运行。每阶段均启动独立 Python 进程，首段要求 CUDA 检测，后两段禁用 CUDA。首次全量运行使用作者开放的 11 个校园共享空间视频、原生时钟、作者轨迹及逐片尺度，检查 3,066 窗、1,656 个历史图像时刻。后两段使用冻结的 PINNS 留一地点权重。请提前挂载下列私有归档，保持文件名。[DUT 作者资料](https://github.com/dongfang-steven-yang/vci-dataset-dut/tree/80b8c746833664cd1e5244fccad79c7f2a7cbe31)

- 个体预测需要 `shortpose_heldout.zip` 或 `shortpose_heldout_five_seeds.zip` 中的一份，可使用此前已有 Kaggle 输出或授权私有副本。
- 图模型需要 `pinns_graph_control.zip`，来自 Kaggle 训练版本 373627962。
- 姿态缓存为可选输入。已有同源、完整 `dut_shared_space_pose_cache.zip` 时，挂载并在下方填它在来源运行中实际记录的 SHA-256；未挂载时第一阶段将真实执行全量提取。该包未预置任何已完成的 DUT 缓存或预报结果。

每个输出含运行记录与私有逐窗数据。保留私有 Kaggle 可见性。

In [ ]:
"""Orchestrate frozen functions in isolated Python processes and verify shared cache before scoring."""
import base64
import hashlib
import io
import json
import os
from pathlib import Path
import re
import shutil
import subprocess
import sys
import tempfile
from zipfile import ZipFile
import zlib

import numpy as np

MOUNT = Path('/kaggle/input')
WORKING = Path('/kaggle/working')
CACHE = WORKING/'dut_shared_space_pose_cache.zip'
PINNED_KEYS = '02c44cf55e5ca8690dc1fe9ac9c076af44dc89f11629b8ccc9e725e49e4580ec'
PINNED_FRAMES = 'f6b71b364a7d0a966cf108dc68a01c60badcf5f0983fea2b0fe6ea9ee02885c5'
POSE_MODEL = '5c0a4bf67953e6d2ac43ce15e77dc9d5d354ae18430a47d2c5963a7bc5683e3c'
CACHE_PROTOCOL = '37aa24490d031d2f6e57fb52fecd21b70070e10775770159f5303638c4b4ed53'
EXTRACTORS = {'4862962a9e6454583aa2ae7593fc232422c428aa251e7b1f2e8a53cc59ff77db',
              '99215e89b42eeb1589da103306bc02efe408b0d0e6a1d7a1ebc9c30b7212ba73'}


def check(truth, reason):
    if not truth:
        raise ValueError(reason)


def sha(data):
    return hashlib.sha256(data).hexdigest()


def file_sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for part in iter(lambda:stream.read(4*1024*1024),b''):
            h.update(part)
    return h.hexdigest()


def roster_sha(entries):
    return sha(json.dumps(entries,separators=(',',':')).encode())


def validate_zip(path):
    with ZipFile(path) as z:
        names=z.namelist()
        check(len(set(names))==len(names) and z.testzip() is None,'Returned ZIP has duplicate names or CRC errors')
        m=json.loads(z.read('sha256.json'))
        check(set(m)|{'sha256.json'}==set(names),'Incomplete original per-member manifest')
        for name,digest in m.items():
            check(sha(z.read(name))==digest,'Invalid member SHA-256: '+name)
        return json.loads(z.read('aggregate.json'))


def verify_full_cache(path, expected=''):
    dig=file_sha(path)
    if expected:
        check(re.fullmatch(r'[0-9a-f]{64}',expected) and expected==dig,'Mounted cache SHA-256 does not match the selected source')
    rep=validate_zip(path)
    check(rep['mode']=='all_windows' and rep['windows']==3066 and rep['source_historical_images']==1656 and
          rep['requested_historical_person_observations']==15330,'Incomplete or pilot cache')
    check(rep['full_cohort_keys_sha256']==PINNED_KEYS and rep['score_protocol_sha256']==CACHE_PROTOCOL and
          rep['script_sha256'] in EXTRACTORS,'Unregistered source protocol/pose extractor')
    check(rep['onnx_sha256']==POSE_MODEL and rep['detector_url'].endswith('/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth') and
          re.fullmatch('dd69338a[0-9a-f]{56}',rep['detector_sha256']),'Detector/pose-model source disagreement')
    with ZipFile(path) as z:
        check(sha(z.read('source.py'))==rep['script_sha256'],'Extractor source is inconsistent')
        with np.load(io.BytesIO(z.read('private.npz')),allow_pickle=False) as v:
            keys=json.loads(str(v['keys_json'].item()));frames=v['source_frame_1based']
            check(len(keys)==3066 and frames.shape==(3066,5) and roster_sha(keys)==PINNED_KEYS and
                  roster_sha(frames.tolist())==PINNED_FRAMES,'Cache row key / frame discrepancy')
            b,k,m=v['bbox'],v['kpts'],v['matched']
            check(m.shape==(3066,5) and m.dtype==bool and b.shape==(3066,5,4) and k.shape==(3066,5,17,3) and
                  np.isfinite(b).all() and np.isfinite(k).all(),'Invalid cache array shape or values')
            check(np.all(k[~m]==0) and np.all(b[~m]==0) and np.all(b[m][:,2:]>0) and int(m.sum())==rep['matched_observations'] and
                  np.bincount(m.sum(1),minlength=6).tolist()==rep['matched_frames_per_window_0_to_5'],'Mismatch in saved detection masks or counts')
            tags=json.loads(str(v['association_json'].item()))
            lookup={(s,int(pid),int(f)):status for s,pid,f,status in tags}
            check(len(tags)==len(lookup)==15330,'Missing/duplicated association records')
            expected_keys={(s,pid,int(f)) for (s,pid,_),group in zip(keys,frames) for f in group}
            valid={'unique','missing','multiple_boxes','multiple_people_in_box','multiple_boxes_and_people'}
            check(set(lookup)==expected_keys and set(lookup.values())<=valid,'Invalid native person-frame association')
            mask=np.array([[lookup[s,pid,int(f)]=='unique' for f in group] for (s,pid,_),group in zip(keys,frames)])
            np.testing.assert_array_equal(m,mask)
    return dig


def run_frozen(packed, expected, label, use_gpu=False):
    check(MOUNT.is_dir() and WORKING.is_dir(),'Run this cell in a private Kaggle notebook')
    code=zlib.decompress(base64.b64decode(packed,validate=True))
    check(sha(code)==expected,'Frozen launcher SHA-256 mismatch: '+label)
    env={**os.environ,'PYTHONDONTWRITEBYTECODE':'1','OPENBLAS_NUM_THREADS':'1','OMP_NUM_THREADS':'2','MKL_NUM_THREADS':'2'}
    if use_gpu:
        env['DUT_DETECTOR_DEVICE']='cuda'
    else:
        env['CUDA_VISIBLE_DEVICES']=''
    with tempfile.TemporaryDirectory(prefix='dut_forecast_') as tmp:
        source=Path(tmp)/'launcher.py';source.write_bytes(code)
        view = MOUNT
        if not list(MOUNT.rglob('pinns_graph_control.zip')) or not list(MOUNT.rglob('shortpose_heldout.zip')):
            st = Path(tmp)/'inputs'; st.mkdir()
            pairs = [('shortpose_heldout.bin','shortpose_heldout.zip','507c0a5decf79a4a48b982987e89c37b8c99e456f5c7f11cc3d49f34bd5741ec'),
                     ('pinns_graph_control.bin','pinns_graph_control.zip','0b3e590b8d7cb2bf99670fd27d3bdbf2a40fa0e77bf29f7ac0423b73535589f8')]
            for bname, zname, exp in pairs:
                raw = (next(MOUNT.rglob(zname)) if list(MOUNT.rglob(zname)) else next(MOUNT.rglob(bname))).read_bytes()
                check(sha(raw) == exp, 'Input SHA mismatch: ' + zname)
                (st/zname).write_bytes(raw)
            view = st
        subprocess.run([sys.executable,'-u','-c',"import builtins, pathlib, runpy, sys, types\nsource, root = pathlib.Path(sys.argv[1]), pathlib.Path(sys.argv[2])\nsys.argv = [str(source)]\nif root == pathlib.Path('/kaggle/input'):\n    runpy.run_path(str(source), run_name='__main__')\nelse:\n    proxy = types.ModuleType('pathlib')\n    proxy.Path = lambda *p: root if pathlib.Path(*p) == pathlib.Path('/kaggle/input') else pathlib.Path(*p)\n    orig_imp = builtins.__import__\n    def imp(name, g=None, l=None, fromlist=(), level=0):\n        if name == 'pathlib' and tuple(fromlist or ()) == ('Path',) and level == 0:\n            return proxy\n        return orig_imp(name, g, l, fromlist, level)\n    ctx = {'__name__': '__main__', '__file__': str(source), '__builtins__': {**vars(builtins), '__import__': imp}}\n    exec(compile(source.read_bytes(), str(source), 'exec'), ctx)\n",str(source),str(view)],cwd=tmp,env=env,check=True)


def restore_or_build(packed, expected_launcher, mounted_sha):
    check(MOUNT.is_dir() and WORKING.is_dir(),'Run this cell in a private Kaggle notebook')
    mounted_sha=mounted_sha.strip().lower()
    found=[p for p in sorted(MOUNT.rglob(CACHE.name)) if p.is_file()]
    if found:
        check(bool(mounted_sha),'A mounted full cache was found. Enter the SHA-256 from its source run in EXISTING_CACHE_SHA256.')
        # The frozen downstream launchers scan all matching mounted filenames; reject conflicting copies before scoring.
        check(all(file_sha(p)==mounted_sha for p in found),'Multiple or unexpected full cache versions mounted')
        verify_full_cache(found[0],mounted_sha)
        if CACHE.exists():
            check(file_sha(CACHE)==mounted_sha,'Working cache differs from selected mounted cache')
        else:
            with tempfile.NamedTemporaryFile(dir=WORKING,prefix='verified_cache_',delete=False) as f:
                tmp=Path(f.name)
                with found[0].open('rb') as src:shutil.copyfileobj(src,f)
            try:
                check(file_sha(tmp)==mounted_sha,'Copy verification failed');tmp.replace(CACHE)
            finally:tmp.unlink(missing_ok=True)
    elif CACHE.exists():
        verify_full_cache(CACHE,mounted_sha)
    else:
        check(not mounted_sha,'A mounted cache digest was supplied but the source ZIP is missing')
        run_frozen(packed,expected_launcher,'DUT full pose extraction',use_gpu=True)
    digest=verify_full_cache(CACHE,mounted_sha)
    receipt=WORKING/'dut_full_pose_sha256.txt'
    receipt.write_text(digest+'  '+CACHE.name+'\n')
    print('Verified 3,066 windows and 1,656 native history images. Cache SHA-256:',digest)
    return digest


## 1　建立或复用完整姿态缓存

首次创建时留空 SHA-256。若挂载以前成功运行的同名完整缓存，请从来源运行日志或 SHA-256 回执复制 64 位摘要。工作区已有缓存亦在重新验证后使用。

In [ ]:
EXISTING_CACHE_SHA256 = ''
FULL_CACHE_SHA = restore_or_build(
(
    '...', EXISTING_CACHE_SHA256)


## 2　个体 M0/M1 前向

验证源权重和完整缓存，单独在 CPU 对相同窗口执行冻结个体模型。该阶段重新从作者资料重建窗口与帧号，用于核对缓存和尺度。

In [ ]:
assert verify_full_cache(CACHE, FULL_CACHE_SHA) == FULL_CACHE_SHA
run_frozen(
(
    '...'
    'dA5l+TPjssww32X5rNiOLH//4ysV/cf/Av683A0='
),
    '0befd550e67377c8bcc0611215d373189117a3cb974fee08703fe9b18df48f9c', 'individual', use_gpu=False)
returned = WORKING/'dut_shared_space_fusion.zip'
report = validate_zip(returned)
assert report['pose_cache_sha256'] == FULL_CACHE_SHA
print(returned, returned.stat().st_size, 'SHA-256', file_sha(returned))


## 3　异类人车图前向

另起 CPU 进程。使用人/车跟踪标注构图，将真实图 M2b 与同批训练的 M1、训练空图及冻结边干预对照；同一完整缓存支持轴反射和保障车辆槽的登记分析。

In [ ]:
assert verify_full_cache(CACHE, FULL_CACHE_SHA) == FULL_CACHE_SHA
run_frozen(
(
    '...', 'graph', use_gpu=False)
returned = WORKING/'dut_shared_space_graph_transfer.zip'
report = validate_zip(returned)
assert report['pose_cache_sha256'] == FULL_CACHE_SHA
print(returned, returned.stat().st_size, 'SHA-256', file_sha(returned))
